### Data Fetching

In [17]:
import psycopg2
import pandas as pd

In [18]:
def fetch_table_to_dataframe(host_ip, database_name, user, password, table_name, port=5432):
    try:
        connection = psycopg2.connect(
            host=host_ip, database=database_name, user=user, password=password, port=port
        )
        print(f"Connected successfully to {database_name} on {host_ip}")
        df = pd.read_sql_query(f"SELECT * FROM {table_name};", connection)
        print(f"✅ Fetched {len(df)} rows from '{table_name}'")
        return df
    except Exception as e:
        print(f"❌ Error: {e}")
        return None
    finally:
        if 'connection' in locals():
            connection.close()

# --- Configuration ---
HOST_IP = "100.94.14.115"
DATABASE_NAME = "pradigma-extractor"
USER = "postgres"
PASSWORD = "password"
PORT = 5432
TABLE_NAME = "extraction"

df_original = fetch_table_to_dataframe(HOST_IP, DATABASE_NAME, USER, PASSWORD, TABLE_NAME, PORT)

Connected successfully to pradigma-extractor on 100.94.14.115


C:\Users\win 11\AppData\Local\Temp\ipykernel_15304\63475477.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(f"SELECT * FROM {table_name};", connection)


✅ Fetched 7912 rows from 'extraction'


In [19]:
keywords = ["WiFi"]

pattern = '|'.join(keywords)

df = df_original.copy(deep=True)
df = df[
    (df['status_id'] == 1) &
    (df['dept_name'] == 'Signalling-And-Communication') &
    (df['filename'].str.contains(pattern, case=False, na=False))
][['filename', 'workorder_id', 'remarks', 'json_data']]

df

,filename,workorder_id,remarks,json_data
6805,SC_PM_NA_WiFi_NA_123.pdf,NaN,,"{'notification': {'notification_no': 'NA', 'no..."
6850,SC_PM_NA_WiFi_NA_9.pdf,NaN,,"{'notification': {'notification_no': 'NA', 'no..."
6970,SC_PM_NA_WiFi_NA_95.pdf,NaN,NOT OCC,"{'notification': {'notification_no': 'NA', 'no..."
6971,SC_PM_NA_WiFi_NA_75.pdf,NaN,,"{'notification': {'notification_no': 'NA', 'no..."
6982,SC_PM_NA_WiFi_NA_94.pdf,NaN,OCC,"{'notification': {'notification_no': 'NA', 'no..."
...,...,...,...,...
7907,SC_PM_NA_WiFi_NA_106.pdf,NaN,,"{'notification': {'notification_no': 'NA', 'no..."
7908,SC_PM_NA_WiFi_NA_104.pdf,NaN,,"{'notification': {'notification_no': 'NA', 'no..."
7909,SC_PM_NA_WiFi_NA_102.pdf,NaN,,"{'notification': {'notification_no': 'NA', 'no..."
7910,SC_PM_NA_WiFi_NA_100.pdf,NaN,,"{'notification': {'notification_no': 'NA', 'no..."


In [20]:
import pandas as pd

valid_json = df['json_data']
valid_json = valid_json[valid_json.apply(lambda x: isinstance(x, dict))]

all_keys = set()
for item in valid_json:
    all_keys.update(item.keys())

print(sorted(all_keys))

['notification', 'wifi', 'work_order']


In [21]:
import pandas as pd
import json

df_wifi = df.copy(deep=True)

valid_mask = df_wifi['json_data'].apply(lambda x: isinstance(x, dict))

rows = []

for _, row in df_wifi[valid_mask].iterrows():
    base_data = {
        'workorder_id': row['workorder_id'],
        'filename': row['filename'],
        'remarks': row['remarks']
    }

    cctv_data = row['json_data'].get('wifi', {})

    for key, value in cctv_data.items():
        if isinstance(value, dict):
            base_data[key] = json.dumps(value)
        else:
            base_data[key] = value

    rows.append(base_data)

df_wifi = pd.DataFrame(rows)

df_wifi = df_wifi.drop(
    columns=["pm_order_no", "reference_document"],
    errors="ignore"
)

df_wifi

,workorder_id,filename,remarks,station,date_time,procedures,comment_recommendation,performed_by,verified_by
0,NaN,SC_PM_NA_WiFi_NA_123.pdf,,BAS,08/12/2024,"{""safety_and_preparation"": {""a"": {""desc"": ""Mak...",NA,22876,7110
1,NaN,SC_PM_NA_WiFi_NA_9.pdf,,IBI,05/06/2022,"{""safety_and_preparation"": {""a"": {""desc"": ""Mak...",OK,7110,7111
2,NaN,SC_PM_NA_WiFi_NA_95.pdf,NOT OCC,BNG,06/12/2023,"{""safety_and_preparation"": {""a"": {""desc"": ""Mak...",All equipments in good condition,19922,7111
3,NaN,SC_PM_NA_WiFi_NA_75.pdf,,HAH,04/09/2023,"{""safety_and_preparation"": {""a"": {""desc"": ""Mak...",all equipment in good condition,SA Team,7111
4,NaN,SC_PM_NA_WiFi_NA_94.pdf,OCC,OCC,12/09/2023,"{""safety_and_preparation"": {""a"": {""desc"": ""Mak...",,7262,7111
...,...,...,...,...,...,...,...,...,...
146,NaN,SC_PM_NA_WiFi_NA_106.pdf,,BNG,06/03/2024,"{""safety_and_preparation"": {""a"": {""desc"": ""Mak...",- All equipment and cabling in good condition,21048,7111
147,NaN,SC_PM_NA_WiFi_NA_104.pdf,,OCC,12/03/2024,"{""safety_and_preparation"": {""a"": {""desc"": ""Mak...",All equipment in good condition,7248,7111
148,NaN,SC_PM_NA_WiFi_NA_102.pdf,,BAS,08/03/2024,"{""safety_and_preparation"": {""a"": {""desc"": ""Mak...",- All hardware in good condition.\n- Cable tig...,20079,7111
149,NaN,SC_PM_NA_WiFi_NA_100.pdf,,TSA,11/03/2024,"{""safety_and_preparation"": {""a"": {""desc"": ""Mak...",All equipment in good condition,7248,7111


In [22]:
df_wifi.columns

Index(['workorder_id', 'filename', 'remarks', 'station', 'date_time',
       'procedures', 'comment_recommendation', 'performed_by', 'verified_by'],
      dtype='object')

In [23]:
def flatten_procedures(proc_data):
    """
    Flattens JSON into 'module.step.status' and 'module.step.remarks' format.
    """
    flat_data = {}
    
    if isinstance(proc_data, str):
        try:
            proc_data = json.loads(proc_data)
        except:
            return {}
            
    if not isinstance(proc_data, dict): 
        return flat_data

    def walk(d, parent_key=''):
        for k, v in d.items():
            new_key = f"{parent_key}.{k}" if parent_key else k
            
            if isinstance(v, dict):
                if 'status' in v:
                    flat_data[f"{new_key}.status"] = v.get('status')
                    flat_data[f"{new_key}.remarks"] = v.get('remarks')
                else:
                    walk(v, new_key)
            else:
                flat_data[new_key] = v

    walk(proc_data)
    return flat_data

def parse_json(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except:
            return {}
    return x

df_wifi["procedures"] = df_wifi["procedures"].apply(parse_json)

df_proc_flat = pd.DataFrame(
    df_wifi["procedures"].apply(flatten_procedures).tolist(), 
    index=df_wifi.index
)

df_final = pd.concat([df_wifi.drop(columns=['procedures']), df_proc_flat], axis=1)

cleanliness_cols = [col for col in df_final.columns if col.startswith('clealiness')]

for col in cleanliness_cols:
    df_final.rename(
        columns={col: col.replace('clealiness', 'cleanliness')},
        inplace=True
    )

print(df_final.columns)

Index(['workorder_id', 'filename', 'remarks', 'station', 'date_time',
       'comment_recommendation', 'performed_by', 'verified_by',
       'safety_and_preparation.a.status', 'safety_and_preparation.a.remarks',
       'safety_and_preparation.b.status', 'safety_and_preparation.b.remarks',
       'power_supply.a.status', 'power_supply.a.remarks',
       'power_supply.b.status', 'power_supply.b.remarks',
       'power_supply.c.status', 'power_supply.c.remarks',
       'power_supply.d.status', 'power_supply.d.remarks',
       'visual_inspection_of_the_equipment.a.status',
       'visual_inspection_of_the_equipment.a.remarks',
       'visual_inspection_of_the_equipment.b.status',
       'visual_inspection_of_the_equipment.b.remarks',
       'visual_inspection_of_the_equipment.c.status',
       'visual_inspection_of_the_equipment.c.remarks',
       'visual_inspection_of_the_equipment.d.status',
       'visual_inspection_of_the_equipment.d.remarks',
       'visual_inspection_of_the_equipment

In [14]:
list(df_final.columns)

['workorder_id',
 'filename',
 'remarks',
 'station',
 'date_time',
 'comment_recommendation',
 'performed_by',
 'verified_by',
 'safety_and_preparation.a.status',
 'safety_and_preparation.a.remarks',
 'safety_and_preparation.b.status',
 'safety_and_preparation.b.remarks',
 'power_supply.a.status',
 'power_supply.a.remarks',
 'power_supply.b.status',
 'power_supply.b.remarks',
 'power_supply.c.status',
 'power_supply.c.remarks',
 'power_supply.d.status',
 'power_supply.d.remarks',
 'visual_inspection_of_the_equipment.a.status',
 'visual_inspection_of_the_equipment.a.remarks',
 'visual_inspection_of_the_equipment.b.status',
 'visual_inspection_of_the_equipment.b.remarks',
 'visual_inspection_of_the_equipment.c.status',
 'visual_inspection_of_the_equipment.c.remarks',
 'visual_inspection_of_the_equipment.d.status',
 'visual_inspection_of_the_equipment.d.remarks',
 'visual_inspection_of_the_equipment.e.status',
 'visual_inspection_of_the_equipment.e.remarks',
 'clealiness_of_the_workstati

In [24]:
output_file = f"../../output/snc/wifi.xlsx"

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_final.to_excel(writer, sheet_name='wifi', index=False)

print(f"Saved excel to {output_file}")

Saved excel to ../../output/snc/wifi.xlsx
